# Notebook 02 — Traitement distribué avec PySpark
## Atelier 2 (1/2) — Préparer PySpark
Session Spark en mode local `local[*]` (tous les cœurs du poste), connecteur MongoDB pour Spark 4 :
`org.mongodb.spark:mongo-spark-connector_2.13:11.1.0` (Scala 2.13, requis par Spark 4.0).

In [ ]:
import os
import sys

# Keep Spark workers on the notebook's Python interpreter.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_HOSTNAME", "localhost")
hadoop_home = os.environ.get("HADOOP_HOME")
if hadoop_home and not os.path.isdir(hadoop_home):
    os.environ.pop("HADOOP_HOME")
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BigDataCommerceAnalytics")
    .master("local[*]")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0")
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark      :", pyspark.__version__)
print("Spark        :", spark.version)
print("Mode         :", spark.sparkContext.master)
print("Default parallelism :", spark.sparkContext.defaultParallelism)
print("Spark UI     :", spark.sparkContext.uiWebUrl)

## Atelier 2 (2/2) — 1. Lire les données depuis MongoDB

In [ ]:
def lire_collection(nom):
    return (spark.read.format("mongodb")
            .option("database", "commerce")
            .option("collection", nom)
            .load())

df_commandes = lire_collection("raw_commandes")
df_payments  = lire_collection("raw_payments")

print("=== raw_commandes ===")
df_commandes.printSchema()
df_commandes.show(5, truncate=False)

print("=== raw_payments ===")
df_payments.printSchema()
df_payments.show(5, truncate=False)

print("Documents lus — commandes :", df_commandes.count(), "| paiements :", df_payments.count())

## 2. Verifier les donnees sources

Avant les transformations, on confirme que MongoDB a fourni les colonnes necessaires aux calculs et a la jointure.

In [ ]:
colonnes_requises_commandes = {
    "cart_id", "order_id", "user_id", "username", "category",
    "product_id", "quantity", "discounted_line_total", "cart_discounted_total"
}
colonnes_requises_paiements = {
    "cart_id", "payment_id", "status", "amount", "amount_collected"
}

for nom, dataframe, colonnes_requises in [
    ("raw_commandes", df_commandes, colonnes_requises_commandes),
    ("raw_payments", df_payments, colonnes_requises_paiements),
]:
    colonnes_manquantes = colonnes_requises - set(dataframe.columns)
    if colonnes_manquantes:
        raise ValueError(f"{nom}: colonnes absentes: {sorted(colonnes_manquantes)}")

print("Schemas verifies : commandes et paiements.")

## 3. Agreger les ventes par categorie

Spark regroupe les lignes de commande par categorie et calcule le nombre de commandes, les quantites vendues et le chiffre d'affaires apres remise.

In [ ]:
df_ventes_categorie = (
    df_commandes
    .groupBy("category")
    .agg(
        F.countDistinct("order_id").alias("nombre_commandes"),
        F.sum("quantity").alias("quantite_vendue"),
        F.round(F.sum("discounted_line_total"), 2).alias("chiffre_affaires"),
    )
    .orderBy(F.desc("chiffre_affaires"))
)

df_ventes_categorie.show(10, truncate=False)

## 4. Analyser les paiements

Cette aggregation compare, pour chaque statut, le nombre de paiements, le montant des commandes et le montant effectivement encaisse.

In [ ]:
df_synthese_paiements = (
    df_payments
    .groupBy("status")
    .agg(
        F.countDistinct("payment_id").alias("nombre_paiements"),
        F.round(F.sum("amount"), 2).alias("montant_commandes"),
        F.round(F.sum("amount_collected"), 2).alias("montant_encaisse"),
    )
    .orderBy("status")
)

df_synthese_paiements.show(truncate=False)

## 5. Reconstituer le suivi par panier

On resume d'abord les lignes de commande au niveau du panier, puis on les joint au paiement correspondant. La jointure gauche conserve aussi les paniers sans paiement associe.

In [ ]:
commandes_par_panier = (
    df_commandes
    .groupBy("cart_id")
    .agg(
        F.first("order_id", ignorenulls=True).alias("order_id"),
        F.first("user_id", ignorenulls=True).alias("user_id"),
        F.first("username", ignorenulls=True).alias("username"),
        F.first("cart_discounted_total", ignorenulls=True).alias("montant_panier"),
        F.countDistinct("product_id").alias("nombre_produits"),
        F.sum("quantity").alias("quantite_totale"),
    )
)

paiements_par_panier = df_payments.select(
    "cart_id", "payment_id", "status", "amount_collected"
)

df_suivi_paniers = commandes_par_panier.join(
    paiements_par_panier,
    on="cart_id",
    how="left",
)

df_suivi_paniers.orderBy("cart_id").show(10, truncate=False)